# Notebook C1 - Cache DermLIP features for fairness datasets (PAD-UFES-20 / Fitzpatrick17k + DDI)

GPU step (run on **Kaggle**, or locally on CPU since these sets are small). Encodes clinical skin images with frozen DermLIP and caches embeddings + zero-shot 7-point concept scores + **skin-tone group** (light/mid/dark) + **malignant** label. The fairness experiment (Notebook C2) then runs on your PC from these caches.

**Datasets (any diverse train set + DDI external):**
- **PAD-UFES-20** -- open access, no request; has Fitzpatrick skin type + malignancy. Use this as the diverse TRAIN set now.
- **Fitzpatrick17k** -- needs access request; drop-in when it arrives (loader is generic).
- **DDI** (`ddidiversedermatologyimages`) -- biopsy-proven, diverse skin tones; the external TEST set.

**Kaggle setup**: GPU on; Internet on; Add Input -> attach whichever of the above you have. Run all. Output -> `/kaggle/working/fairness_cache/`; Save Version -> Create Dataset -> feed to Notebook C2. Cells for missing datasets simply print 'skipping'.

Note: public mirrors vary in column names / folder layout. The loaders print columns and unique values and use defensive auto-detection; adjust the small config maps if your copy differs.

In [ ]:
!pip install -q open_clip_torch
print('open_clip installed')

In [ ]:
# --- config + model ---
import os, glob, json, time
import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEV_TYPE = 'cuda' if DEVICE == 'cuda' else 'cpu'
MODEL_NAME = 'hf-hub:redlessone/DermLIP_ViT-B-16'
BATCH_SIZE = 32
OUT_DIR = '/kaggle/working/fairness_cache'
SEARCH = ['/kaggle/input', '.', '..']
os.makedirs(OUT_DIR, exist_ok=True)

print('Loading DermLIP ...')
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False
print('Device:', DEVICE)

In [ ]:
# --- prompts + encode helpers (shared) ---
CONCEPT_PROMPTS = {
    'atypical_pigment_network': ('a skin lesion with an atypical pigment network', 'a skin lesion with a typical pigment network'),
    'blue_whitish_veil': ('a skin lesion with a blue-whitish veil', 'a skin lesion without a blue-whitish veil'),
    'atypical_vascular': ('a skin lesion with atypical vascular structures', 'a skin lesion with regular vascular structures'),
    'irregular_streaks': ('a skin lesion with irregular streaks', 'a skin lesion without irregular streaks'),
    'irregular_pigmentation': ('a skin lesion with irregular pigmentation', 'a skin lesion with regular pigmentation'),
    'irregular_dots_globules': ('a skin lesion with irregular dots and globules', 'a skin lesion with regular dots and globules'),
    'regression': ('a skin lesion with regression structures', 'a skin lesion without regression structures'),
}
MALIG_PROMPTS = ('a photo of a malignant skin lesion', 'a photo of a benign skin lesion')

def encode_images(paths):
    n = len(paths)
    feats, order = [], []
    i = 0
    while i < n:
        tens, idxs = [], []
        for j, p in enumerate(paths[i:i + BATCH_SIZE]):
            try:
                tens.append(preprocess(Image.open(p).convert('RGB')))
                idxs.append(i + j)
            except Exception:
                pass
        if tens:
            x = torch.stack(tens).to(DEVICE)
            with torch.no_grad(), torch.autocast(device_type=DEV_TYPE, enabled=(DEV_TYPE == 'cuda')):
                f = model.encode_image(x)
                f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.float().cpu().numpy())
            order.extend(idxs)
        i += BATCH_SIZE
    feats = np.concatenate(feats, axis=0) if feats else np.zeros((0, 512), np.float32)
    dd = feats.shape[1] if feats.shape[0] else 512
    emb = np.zeros((n, dd), dtype=np.float32)
    mask = np.zeros(n, dtype=bool)
    for k, idx in enumerate(order):
        emb[idx] = feats[k]; mask[idx] = True
    return emb, mask

def encode_text(prompts):
    tok = tokenizer(list(prompts)).to(DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEV_TYPE, enabled=(DEV_TYPE == 'cuda')):
        t = model.encode_text(tok)
        t = t / t.norm(dim=-1, keepdim=True)
    return t.float().cpu().numpy()

def softmax2(a, b):
    z = np.stack([a, b], axis=1) * 100.0
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def concept_scores(emb, mask):
    names = list(CONCEPT_PROMPTS.keys())
    pres = encode_text([CONCEPT_PROMPTS[k][0] for k in names])
    absn = encode_text([CONCEPT_PROMPTS[k][1] for k in names])
    out = np.zeros((emb.shape[0], len(names)), dtype=np.float32)
    for c in range(len(names)):
        out[:, c] = softmax2(absn[c] @ emb.T, pres[c] @ emb.T)[:, 1]
    out[~mask] = np.nan
    return out, names

def malig_prob(emb, mask):
    t = encode_text(list(MALIG_PROMPTS))
    out = softmax2(t[1] @ emb.T, t[0] @ emb.T)[:, 1]
    out[~mask] = np.nan
    return out

def find_one(name, roots=SEARCH):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, '**', name), recursive=True))
        if hits:
            return hits[0]
    return None

def cache_dataset(tag, paths, malignant, group, split):
    print('[' + tag + '] encoding', len(paths), 'images ...')
    t0 = time.time()
    emb, mask = encode_images(paths)
    cs, cnames = concept_scores(emb, mask)
    mp = malig_prob(emb, mask)
    print('  ok:', int(mask.sum()), '/', len(paths), 'in', round(time.time() - t0, 1), 's')
    np.savez_compressed(os.path.join(OUT_DIR, 'features_' + tag + '.npz'),
        emb=emb, mask=mask, concept_scores=cs,
        malignant=np.asarray(malignant, dtype=np.int64),
        group=np.asarray(group, dtype='U8'), split=np.asarray(split, dtype='U8'),
        malig_prob=mp)
    pd.DataFrame({'malignant': malignant, 'group': group, 'split': split}).to_parquet(
        os.path.join(OUT_DIR, 'meta_' + tag + '.parquet'))
    return cnames

In [ ]:
# --- Fitzpatrick17k (16.5k clinical photos, Fitzpatrick I-VI; diverse TRAIN set) ---
# Labels CSV is found locally if attached, else auto-downloaded from the official repo
# (so you only need to attach an IMAGES-only mirror, e.g. mobaswiralfarabi/fitzpatrick17k_original).
fcsv = None
for r in SEARCH:
    for cand in sorted(glob.glob(os.path.join(r, '**', '*.csv'), recursive=True)):
        try:
            cols = pd.read_csv(cand, nrows=1).columns
        except Exception:
            continue
        if any('fitzpatrick' in c.lower() for c in cols) and any(p in cols for p in ['three_partition_label', 'nine_partition_label']):
            fcsv = cand; break
    if fcsv:
        break

if fcsv is not None:
    print('fitz csv (local):', fcsv); f = pd.read_csv(fcsv)
else:
    url = 'https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv'
    try:
        f = pd.read_csv(url); print('fitz csv: downloaded from official GitHub')
    except Exception as e:
        f = None; print('Fitzpatrick17k csv not local and download failed (enable Internet?):', e)

if f is None:
    print('skipping Fitzpatrick17k.')
else:
    print('columns:', list(f.columns))
    tone_col = next((c for c in ['fitzpatrick_scale', 'fitzpatrick', 'fitzpatrick_centaur'] if c in f.columns), None)
    part_col = next((c for c in ['three_partition_label', 'nine_partition_label'] if c in f.columns), None)
    id_col = next((c for c in ['md5hash', 'hasher', 'image', 'filename'] if c in f.columns), None)
    print('using tone/part/id cols:', tone_col, part_col, id_col)
    print('tone values:', sorted([str(v) for v in f[tone_col].dropna().unique()]))
    print('partition values:', sorted([str(v) for v in f[part_col].dropna().unique()]))

    # index every image file once (mirror stores images flat or in subfolders, named by md5hash)
    img_index = {}
    for r in set(SEARCH + [os.path.dirname(fcsv) if fcsv else '.']):
        for ext in ('*.jpg', '*.png', '*.jpeg'):
            for p in glob.glob(os.path.join(r, '**', ext), recursive=True):
                img_index.setdefault(os.path.splitext(os.path.basename(p))[0], p)
    def fpath(h):
        return img_index.get(str(h), str(h))
    def tone_group(v):
        try:
            s = int(float(v))
        except Exception:
            return 'unknown'
        if s in (1, 2): return 'light'
        if s in (3, 4): return 'mid'
        if s in (5, 6): return 'dark'
        return 'unknown'

    f = f[f[part_col].notna() & f[tone_col].notna()].reset_index(drop=True)
    malignant = (f[part_col].astype(str).str.lower() == 'malignant').astype(int).tolist()
    group = [tone_group(v) for v in f[tone_col]]
    paths = [fpath(h) for h in f[id_col]]
    n_found = sum(1 for p in paths if os.path.exists(p))
    print('images matched on disk:', n_found, '/', len(paths))
    rng = np.random.default_rng(0)
    u = rng.random(len(f))
    split = np.where(u < 0.7, 'train', np.where(u < 0.85, 'val', 'test')).tolist()
    cache_dataset('fitz', paths, malignant, group, split)
    print('fitz group counts:', {g: group.count(g) for g in ['light', 'mid', 'dark', 'unknown']})
    print('fitz malignant:', int(np.sum(malignant)), '/', len(malignant))

In [ ]:
# --- DDI (biopsy-proven, diverse skin tones; external test) ---
dcsv = find_one('ddi_metadata.csv') or find_one('ddi.csv')
if dcsv is None:
    print('DDI csv not found - skipping. Attach the DDI dataset and re-run.')
else:
    print('ddi csv:', dcsv)
    dd = pd.read_csv(dcsv)
    print('columns:', list(dd.columns))
    file_col = next((c for c in ['DDI_file', 'file', 'filename', 'image'] if c in dd.columns), None)
    tone_col = next((c for c in ['skin_tone', 'fitzpatrick', 'tone'] if c in dd.columns), None)
    mal_col = next((c for c in ['malignant', 'label'] if c in dd.columns), None)
    print('using file/tone/mal cols:', file_col, tone_col, mal_col)
    print('tone values:', sorted([str(v) for v in dd[tone_col].dropna().unique()]))

    ddir = os.path.dirname(dcsv)
    def dpath(fn):
        cand = os.path.join(ddir, str(fn))
        if os.path.exists(cand):
            return cand
        hit = find_one(str(fn), [ddir])
        return hit if hit else cand
    def ddi_group(v):
        s = str(v).strip()
        if s in ('12', '1', '2'): return 'light'
        if s in ('34', '3', '4'): return 'mid'
        if s in ('56', '5', '6'): return 'dark'
        return 'unknown'

    malignant = dd[mal_col].astype(str).str.lower().isin(['true', '1', 'malignant', 'yes']).astype(int).tolist()
    group = [ddi_group(v) for v in dd[tone_col]]
    paths = [dpath(fn) for fn in dd[file_col]]
    split = ['test'] * len(dd)
    cache_dataset('ddi', paths, malignant, group, split)
    print('ddi group counts:', {g: group.count(g) for g in ['light', 'mid', 'dark', 'unknown']})
    print('ddi malignant:', int(np.sum(malignant)), '/', len(malignant))

In [ ]:
# --- PAD-UFES-20 (open access; Fitzpatrick skin type + malignancy; diverse TRAIN set) ---
# Use this in place of Fitzpatrick17k while access is pending. Identified by its unique
# (img_id + diagnostic) columns so it is not confused with the other datasets.
pcsv = None
for r in SEARCH:
    for cand in sorted(glob.glob(os.path.join(r, '**', '*.csv'), recursive=True)):
        try:
            cols = pd.read_csv(cand, nrows=1).columns
        except Exception:
            continue
        if 'img_id' in cols and 'diagnostic' in cols:
            pcsv = cand; break
    if pcsv:
        break
if pcsv is None:
    print('PAD-UFES-20 metadata not found - skipping (attach the dataset to use it).')
else:
    print('pad csv:', pcsv)
    pdf = pd.read_csv(pcsv)
    tone_col = next((c for c in ['fitspatrick', 'fitzpatrick', 'fitspatrick_scale'] if c in pdf.columns), None)
    print('using tone col:', tone_col, '| diagnostic values:', sorted([str(v) for v in pdf['diagnostic'].dropna().unique()]))
    root = os.path.dirname(pcsv)
    img_index = {}
    for r in set([root] + SEARCH):
        for ext in ('*.png', '*.jpg', '*.jpeg'):
            for p in glob.glob(os.path.join(r, '**', ext), recursive=True):
                img_index.setdefault(os.path.basename(p), p)
    MAL = {'BCC', 'MEL', 'SCC'}    # malignant classes in PAD-UFES-20
    def tone_group(v):
        try:
            s = int(float(v))
        except Exception:
            return 'unknown'
        if s in (1, 2): return 'light'
        if s in (3, 4): return 'mid'
        if s in (5, 6): return 'dark'
        return 'unknown'
    if tone_col is None:
        print('no Fitzpatrick column in PAD metadata - skipping.')
    else:
        pdf = pdf[pdf[tone_col].notna()].reset_index(drop=True)
        malignant = pdf['diagnostic'].astype(str).str.upper().isin(MAL).astype(int).tolist()
        group = [tone_group(v) for v in pdf[tone_col]]
        paths = [img_index.get(str(b), os.path.join(root, str(b))) for b in pdf['img_id']]
        rng = np.random.default_rng(0)
        u = rng.random(len(pdf))
        split = np.where(u < 0.7, 'train', np.where(u < 0.85, 'val', 'test')).tolist()
        cache_dataset('pad', paths, malignant, group, split)
        print('pad group counts:', {g: group.count(g) for g in ['light', 'mid', 'dark', 'unknown']})
        print('pad malignant:', int(np.sum(malignant)), '/', len(malignant))

In [ ]:
# --- manifest + sanity ---
from numpy import load as npload
manifest = {'model': MODEL_NAME, 'concept_names': list(CONCEPT_PROMPTS.keys()), 'created': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w') as fh:
    json.dump(manifest, fh, indent=2)
for tag in ['fitz', 'pad', 'ddi']:
    fpz = os.path.join(OUT_DIR, 'features_' + tag + '.npz')
    if os.path.exists(fpz):
        z = npload(fpz, allow_pickle=True)
        grp = z['group'].astype('U8')
        print(tag, 'emb', z['emb'].shape, 'valid', int(z['mask'].sum()), 'malig', int(z['malignant'].sum()),
              'groups', {g: int((grp == g).sum()) for g in ['light', 'mid', 'dark']})
print('saved to', OUT_DIR, '->', os.listdir(OUT_DIR))

## Next
Save Version -> Create Dataset from `/kaggle/working/fairness_cache` -> attach to **Notebook C2** (Group-DRO fairness + faithfulness-by-skin-tone), which runs on your PC (CPU).